# Assignment 2, notebook 1: Exploratory data analysis (Part A)
**Goal:** understand the FY2014-2019 loans before building a bad-loan model.
**Bad loan** = a loan with a charge-off amount above zero. We use the amount rather than the status label, because the label misses some losses.
This notebook covers data quality, distributions, default patterns and the time lag in losses.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

TEAL, GREY, RED ="#0E7C86", "#B8BFCC", "#C0392B"
OUT = Path("../outputs"); OUT.mkdir(parents=True, exist_ok=True)

In [2]:
# Load and scope

df = pd.read_parquet("../../data/processed/loans_clean.parquet")
d = df[df["approvalfy"].between(2014, 2019)].copy()
d["bad"] = (d["grosschargeoffamount"].fillna(0) > 0).astype(int)

print(f"All cleaned loans: {len(df):,} | FY2014-2019: {len(d):,} | bad-loan rate: {d['bad'].mean():.2%}")
d.groupby("approvalfy").agg(loans=("bad", "size"), bad_rate=("bad", "mean"),
                            gross_usd_bn=("grossapproval", lambda s: s.sum() / 1e9)).round(3)

All cleaned loans: 478,650 | FY2014-2019: 313,916 | bad-loan rate: 7.64%


,loans,bad_rate,gross_usd_bn
approvalfy,,,
2014,45908,0.072,16.845
2015,55372,0.077,20.315
2016,56745,0.079,21.595
2017,56062,0.081,23.048
2018,54172,0.081,22.807
2019,45657,0.066,20.551


In [3]:
# Missing values

miss = (d.isna().mean() * 100).round(1).rename("missing_%").to_frame()
miss["n_unique"] = d.nunique()
miss = miss[miss["missing_%"] > 0].sort_values("missing_%", ascending=False)
display(miss)

fig = px.bar(miss.reset_index(), x="missing_%", y="index", orientation="h", color_discrete_sequence=[TEAL],
             title="Share of loans with a blank value, by column (%)", labels={"index": ""}, template="plotly_white")
fig.update_yaxes(autorange="reversed")
fig.show()

,missing_%,n_unique
bankncuanumber,97.0,358
chargeoffdate,92.4,2532
franchisecode,89.5,3770
franchisename,89.5,3538
soldsecmrktind,71.5,2
paidinfulldate,22.6,148
bankfdicnumber,6.4,1915
businessage,0.5,10
firstdisbursementdate,0.1,2356


In [4]:
# Validity checks

checks = {
    "Term of 0 months": int((d["terminmonths"] == 0).sum()),
    "Interest rate of 0": int((d["initialinterestrate"] == 0).sum()),
    "Guarantee outside 0-100%": int(((d["guarantee_pct"] <= 0) | (d["guarantee_pct"] > 1)).sum()),
    "Loan amount <= 0": int((d["grossapproval"] <= 0).sum()),
    "Charge-off dated before approval": int((d["chargeoffdate"] < d["approvaldate"]).sum()),
    "Jobs supported above 200": int((d["jobssupported"] > 200).sum()),
    "Business age = 'Unanswered'": int((d["businessage"] == "Unanswered").sum()),
    "Business age blank": int(d["businessage"].isna().sum()),
}

pd.Series(checks, name="rows").to_frame()

,rows
Term of 0 months,128
Interest rate of 0,7
Guarantee outside 0-100%,0
Loan amount <= 0,0
Charge-off dated before approval,0
Jobs supported above 200,251
Business age = 'Unanswered',30101
Business age blank,1620


In [5]:
# The target, checked against the status label

display(pd.crosstab(d["loanstatus"], d["bad"].map({0: "not bad", 1: "bad"}), margins=True))

print(f"Bad by charge-off amount: {d['bad'].mean():.2%} | status CHGOFF only: {(d['loanstatus'] == 'CHGOFF').mean():.2%}")
print(f"Bad loans whose status is NOT CHGOFF/CLSLN: {((d['bad'] == 1) & ~d['loanstatus'].isin(['CHGOFF', 'CLSLN'])).sum():,}")
loss = d.loc[d["bad"] == 1, "grosschargeoffamount"]
print(f"Charge-off amount: median ${loss.median():,.0f}, total ${loss.sum() / 1e9:.2f}bn; "
      f"share of approved dollars lost: {loss.sum() / d['grossapproval'].sum():.2%}")

bad,bad,not bad,All
loanstatus,,,
CHGOFF,21222,9,21231
CLSLN,785,0,785
CURR,0,40449,40449
DEFERD,0,104,104
DELINQ,0,993,993
LIQUID,1,2448,2449
P I F,1985,241025,243010
PSTDUE,0,511,511
PURCH(NOT C/O),4,4380,4384


Bad by charge-off amount: 7.64% | status CHGOFF only: 6.76%
Bad loans whose status is NOT CHGOFF/CLSLN: 1,990
Charge-off amount: median $55,150, total $3.42bn; share of approved dollars lost: 2.73%


In [6]:
# Numeric distributions

num_cols = {"grossapproval": "Loan amount ($, log scale)", "terminmonths": "Term (months)",
            "initialinterestrate": "Interest rate (%)", "guarantee_pct": "SBA guarantee share",
            "jobssupported": "Jobs supported", "approvalfy": "Approval year"}
display(d[list(num_cols)].describe(percentiles=[.01, .1, .5, .9, .99]).round(2).T)

fig = make_subplots(rows=2, cols=3, subplot_titles=list(num_cols.values()))
for i, c in enumerate(num_cols):
    x = np.log10(d[c].clip(lower=1)) if c == "grossapproval" else d[c]
    fig.add_trace(go.Histogram(x=x, nbinsx=40, marker_color=TEAL, showlegend=False), row=i // 3 + 1, col=i % 3 + 1)
fig.update_xaxes(tickvals=[3, 4, 5, 6], ticktext=["$1k", "$10k", "$100k", "$1m"], row=1, col=1)
fig.update_layout(height=550, template="plotly_white", title="Distributions of the main numeric fields")
fig.show()

,count,mean,std,min,1%,10%,50%,90%,99%,max
grossapproval,313916.0,398706.78,714311.59,1000.0,5000.00,20000.0,132000.00,1091050.00,3820000.00,5000000.0
terminmonths,313916.0,123.51,79.53,0.0,11.00,60.0,96.00,300.00,306.00,432.0
initialinterestrate,313916.0,6.69,1.66,0.0,3.75,5.0,6.25,9.25,11.49,13.5
guarantee_pct,313916.0,0.64,0.14,0.1,0.50,0.5,0.75,0.85,0.85,0.9
jobssupported,313916.0,10.05,18.58,0.0,0.00,0.0,4.00,25.00,94.00,314.0
approvalfy,313916.0,2016.49,1.64,2014.0,2014.00,2014.0,2016.00,2019.00,2019.00,2019.0


In [7]:
# Bad-loan rate by approval year

yr = d.groupby("approvalfy").agg(loans=("bad", "size"), bad_rate=("bad", "mean")).reset_index()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=yr["approvalfy"], y=yr["loans"], name="Loans approved", marker_color=GREY), secondary_y=False)
fig.add_trace(go.Scatter(x=yr["approvalfy"], y=yr["bad_rate"] * 100, name="Bad-loan rate (%)", mode="lines+markers+text",
                         text=(yr["bad_rate"] * 100).round(1), textposition="top center",
                         line=dict(color=RED, width=3)), secondary_y=True)
fig.update_yaxes(title_text="Loans", secondary_y=False)
fig.update_yaxes(title_text="Bad-loan rate (%)", range=[0, 10], secondary_y=True)
fig.update_layout(template="plotly_white", title="Loans approved and bad-loan rate by approval year", height=420)
fig.show()

In [8]:
# Helper and size/term bands

overall = d["bad"].mean()

def rate_by(col, title, order=None, min_n=500, top=None, height=400):
    t = d.groupby(col, observed=True)["bad"].agg(loans="size", bad_rate="mean").reset_index()
    t = t[t["loans"] >= min_n]
    t = t.set_index(col).loc[order].reset_index() if order else t.sort_values("bad_rate", ascending=False)
    if top: t = t.head(top)
    fig = px.bar(t, x=col, y=t["bad_rate"] * 100, text=(t["bad_rate"] * 100).round(1),
                 hover_data={"loans": ":,"}, color_discrete_sequence=[TEAL], title=title, template="plotly_white")
    fig.add_hline(y=overall * 100, line_dash="dot", line_color=RED, annotation_text=f"Overall {overall:.1%}")
    fig.update_traces(textposition="outside")
    fig.update_layout(height=height, yaxis_title="Bad-loan rate (%)", xaxis_title="")
    fig.show()
    return t

d["size_band"] = pd.cut(d["grossapproval"], [0, 50e3, 150e3, 350e3, 1e6, 5e6],
                        labels=["Up to $50k", "$50-150k", "$150-350k", "$350k-1m", "Over $1m"])
d["term_band"] = pd.cut(d["terminmonths"], [-1, 60, 84, 120, 240, 500],
                        labels=["Up to 5y", "5-7y", "7-10y", "10-20y", "Over 20y"])
rate_by("size_band", "Bad-loan rate by loan size", order=list(d["size_band"].cat.categories))
rate_by("term_band", "Bad-loan rate by loan term", order=list(d["term_band"].cat.categories))

,term_band,loans,bad_rate
0,Up to 5y,57605,0.221196
1,5-7y,91672,0.055917
2,7-10y,94595,0.054633
3,10-20y,26395,0.021898
4,Over 20y,43649,0.008775


In [9]:
# Product, business and loan structure

d["interest_type"] = d["fixedorvariableinterestind"].map({"F": "Fixed", "V": "Variable"})
d["collateral"] = d["collateralind"].map({True: "Has collateral", False: "No collateral"})
d["revolver"] = d["revolverstatus"].map({True: "Revolving line", False: "Term loan"})

for col, title in [("subprogram", "Bad-loan rate by SBA product"),
                   ("businessage", "Bad-loan rate by business age"),
                   ("businesstype", "Bad-loan rate by business type"),
                   ("interest_type", "Bad-loan rate by interest type"),
                   ("collateral", "Bad-loan rate by collateral"),
                   ("revolver", "Bad-loan rate by loan structure")]:
    display(rate_by(col, title, min_n=1000).round(3))

,subprogram,loans,bad_rate
0,Community Advantage Initiative,4625,0.143
6,Lender Advantage Initiative,23298,0.098
2,FA$TRK (Small Loan Express),152289,0.084
3,Guaranty,129018,0.063
5,"International Trade - Sec, 7(a) (16)",1099,0.059
12,Standard Asset Based,1804,0.033


,businessage,loans,bad_rate
3,Less than 3 years old but at least 2,15125,0.100
7,"New, Less than 1 Year old",29983,0.098
4,Less than 4 years old but at least 3,12224,0.092
5,Less than 5 years old but at least 4,10906,0.090
8,"Startup, Loan Funds will Open Business",49912,0.086
9,Unanswered,30101,0.078
1,Existing or more than 2 years old,45714,0.073
2,"Existing, 5 or more years",108734,0.060
0,Change of Ownership,9571,0.043


,businesstype,loans,bad_rate
1,INDIVIDUAL,30660,0.085
0,CORPORATION,277951,0.076
2,PARTNERSHIP,5291,0.042


,interest_type,loans,bad_rate
1,Variable,243028,0.084
0,Fixed,70888,0.049


,collateral,loans,bad_rate
1,No collateral,85181,0.112
0,Has collateral,228735,0.063


,revolver,loans,bad_rate
0,Revolving line,95882,0.090
1,Term loan,218034,0.071


In [10]:
# Industry and state

d["sector"] = d["naicscode"].astype("Int64").astype(str).str[:2]
rate_by("sector", "Bad-loan rate by industry sector (NAICS 2-digit, sectors with 2,000+ loans)", min_n=2000, height=450)

st = d.groupby("borrstate")["bad"].agg(loans="size", bad_rate="mean").query("loans >= 2000").sort_values("bad_rate")
ends = pd.concat([st.head(8), st.tail(8)]).reset_index()
fig = px.bar(ends, x="borrstate", y=ends["bad_rate"] * 100, color_discrete_sequence=[TEAL],
             title="States with the lowest and highest bad-loan rates (2,000+ loans)", template="plotly_white")
fig.update_layout(yaxis_title="Bad-loan rate (%)", xaxis_title="")
fig.show()

In [11]:
# Size and term together

heat = d.pivot_table(index="size_band", columns="term_band", values="bad", aggfunc="mean", observed=True) * 100
cnt = d.pivot_table(index="size_band", columns="term_band", values="bad", aggfunc="size", observed=True)
px.imshow(heat.round(1), text_auto=True, color_continuous_scale="Reds", aspect="auto",
          title="Bad-loan rate (%) by loan size and term", template="plotly_white").show()
cnt

term_band,Up to 5y,5-7y,7-10y,10-20y,Over 20y
size_band,,,,,
Up to $50k,31381,51080,17182,1523,29
$50-150k,16942,26414,28520,5169,1089
$150-350k,5815,11042,23938,7555,5627
$350k-1m,2201,2331,17591,7399,17164
Over $1m,1266,805,7364,4749,19740


In [12]:
# Lenders

lt = d.groupby("bankname")["bad"].agg(loans="size", bad_rate="mean").reset_index()
big = lt[lt["loans"] >= 200]
print(f"{len(lt):,} lenders; {len(big)} have 200+ loans. Bad-loan rate across them: "
      f"10th pct {big['bad_rate'].quantile(.1):.1%}, median {big['bad_rate'].median():.1%}, 90th pct {big['bad_rate'].quantile(.9):.1%}")

fig = px.scatter(big, x="loans", y=big["bad_rate"] * 100, log_x=True, hover_name="bankname",
                 color_discrete_sequence=[TEAL], labels={"y": "Bad-loan rate (%)", "loans": "Loans (log scale)"},
                 title="Lender bad-loan rate vs lender size (lenders with 200+ loans)", template="plotly_white")
fig.add_hline(y=overall * 100, line_dash="dot", line_color=RED)
fig.show()

top10 = d["bankname"].value_counts().head(10).index
lt[lt["bankname"].isin(top10)].sort_values("loans", ascending=False).assign(bad_rate=lambda x: (x["bad_rate"] * 100).round(1))

2,256 lenders; 197 have 200+ loans. Bad-loan rate across them: 10th pct 1.4%, median 4.4%, 90th pct 10.5%


,bankname,loans,bad_rate
2194,Wells Fargo Bank National Association,28699,10.4
1968,The Huntington National Bank,24296,6.5
1122,"JPMorgan Chase Bank, National Association",17435,8.7
2074,"U.S. Bank, National Association",14376,8.0
1826,"TD Bank, National Association",11953,16.7
1453,"PNC Bank, National Association",8850,10.3
1249,Manufacturers and Traders Trust Company,8191,8.7
366,Celtic Bank Corporation,6164,20.0
1218,Live Oak Banking Company,5124,2.2
624,Eastern Bank,4771,9.0


In [13]:
# How long losses take (maturity)

d["months_to_co"] = (d["chargeoffdate"] - d["approvaldate"]).dt.days / 30.44
d["age_months"] = (pd.Timestamp("2026-03-31") - d["approvaldate"]).dt.days / 30.44
bad = d[d["bad"] == 1]

print(f"Median months from approval to charge-off: {bad['months_to_co'].median():.0f}; "
      f"share of charge-offs by month 36/60/84: " + " / ".join(f"{(bad['months_to_co'] <= m).mean():.0%}" for m in [36, 60, 84]))
print(f"Loan age at 31 Mar 2026: FY2014 median {d.loc[d.approvalfy == 2014, 'age_months'].median():.0f} months, "
      f"FY2019 {d.loc[d.approvalfy == 2019, 'age_months'].median():.0f} months")

fig = go.Figure()
for fy in range(2014, 2020):
    x = np.sort(bad.loc[bad["approvalfy"] == fy, "months_to_co"].dropna().values)
    fig.add_trace(go.Scatter(x=x, y=np.arange(1, len(x) + 1) / len(x), mode="lines", name=f"FY{fy}"))
fig.update_layout(template="plotly_white", height=420, xaxis_title="Months from approval to charge-off",
                  yaxis_title="Cumulative share", title="How quickly bad loans are charged off, by approval year")
fig.show()

# Same-length window for every year: fair comparison across vintages
d["bad_60m"] = ((d["bad"] == 1) & (d["months_to_co"] <= 60)).astype(int)
d.groupby("approvalfy")[["bad", "bad_60m"]].mean().mul(100).round(2).rename(
    columns={"bad": "ever charged off %", "bad_60m": "charged off within 60 months %"})

Median months from approval to charge-off: 53; share of charge-offs by month 36/60/84: 25% / 59% / 85%
Loan age at 31 Mar 2026: FY2014 median 143 months, FY2019 84 months


,ever charged off %,charged off within 60 months %
approvalfy,,
2014,7.19,4.02
2015,7.65,4.43
2016,7.90,4.42
2017,8.14,4.73
2018,8.14,5.00
2019,6.59,4.16


In [14]:
# Correlations

num = d[["grossapproval", "terminmonths", "initialinterestrate", "guarantee_pct", "jobssupported", "approvalfy", "bad"]].copy()
num["grossapproval"] = np.log10(num["grossapproval"])
num = num.rename(columns={"grossapproval": "log10 loan amount"})
corr = num.corr(method="spearman").round(2)
px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
          title="Rank correlations (Spearman)", template="plotly_white").show()
corr["bad"].drop("bad").sort_values()

terminmonths          -0.29
log10 loan amount     -0.09
jobssupported         -0.03
guarantee_pct         -0.00
approvalfy            -0.00
initialinterestrate    0.13
Name: bad, dtype: float64

In [16]:
d["whole_year_term"] = d["terminmonths"] % 12 == 0
print(f"Whole-year term: {d.loc[d.bad == 0, 'whole_year_term'].mean():.0%} of good loans vs {d.loc[d.bad == 1, 'whole_year_term'].mean():.0%} of bad loans")
d.groupby("whole_year_term")["bad"].agg(loans="size", bad_rate="mean").round(3)

Whole-year term: 86% of good loans vs 10% of bad loans


,loans,bad_rate
whole_year_term,,
False,62832,0.343
True,251084,0.010


In [15]:
# Save key facts

findings = {"fy2014_19_loans": len(d), "bad_rate": round(d["bad"].mean(), 4), "bad_loans": int(d["bad"].sum()),
            "median_months_to_chargeoff": round(bad["months_to_co"].median(), 1), "lenders": int(d["bankname"].nunique())}
pd.Series(findings).to_csv(OUT / "a2_eda_key_facts.csv", header=["value"])
print(findings)

{'fy2014_19_loans': 313916, 'bad_rate': np.float64(0.0764), 'bad_loans': 23997, 'median_months_to_chargeoff': np.float64(53.3), 'lenders': 2256}


## Summary

**Data and target**
- 313,916 loans (FY2014-2019); 23,997 are bad, a 7.64% rate. Charge-offs total about $3.4bn, 2.7% of approved dollars.
- Using the charge-off amount finds about 2,000 bad loans whose status is not CHGOFF/CLSLN, so we use the amount, not the label.
- Data quality is good. Blank columns are mostly expected (NCUA number, charge-off date, franchise, secondary-market flag). Business age is "Unanswered" for about 10%. A handful of rows have term 0 or rate 0.

**What drives bad loans**
- **Term looks like the strongest signal (22% bad for 5 years or less, under 1% over 20 years), but the field is unreliable.** 86% of good loans have a whole-year term against 10% of charged-off loans, which suggests it is revised after approval. We do not use it as a feature.
- **Size:** 10.0% for loans up to $50k, falling to 3.0% over $1m.
- **No collateral** 11.2% vs 6.3% with collateral. **Variable rate** 8.4% vs 4.9% fixed. **Revolving lines** 9.0% vs 7.1% term loans.
- **Business age:** established businesses (5+ years) 6.0%; new or young businesses 8.6-10%.
- **Product:** Community Advantage 14.3%, Express (FA$TRK) 8.4%, standard Guaranty 6.3%.
- **Industry and place:** transport (10.5%) and retail, arts and recreation are high; agriculture (3.7%), finance and health care (about 4.5%) are low. Florida (11.7%) and Texas (10.7%) are high; Idaho and Washington are low.
- **Lender:** among lenders with 200+ loans, the bad-loan rate ranges from 1.4% (10th percentile) to 10.5% (90th), median 4.4%. Lender quality matters a lot, which affects how we validate the model.

**Timing**
- Median time from approval to charge-off is 53 months; about 25% of charge-offs happen by month 36, 59% by 60 and 85% by 84.
- FY2019 loans are about 84 months old and FY2014 loans about 143, so older years have had longer to fail. We will check results with a fixed 60-month window.

**What this means for modelling**
- Use only information known at approval (no status, dates or amounts recorded afterwards).
- Validate across time (train on earlier years, test on later ones).
- Treat lender as a risky feature: test the model on lenders it has not seen, to check it generalises.
- Features to build next: term bands, size on a log scale, collateral, rate type, business age, industry sector and state.